In [1]:
import os
import re
import json
import torch
import pandas as pd
from datetime import datetime
from datasets import Dataset
from transformers import (
    Trainer, TrainingArguments, DataCollatorForSeq2Seq,
    T5Config, T5ForConditionalGeneration, PreTrainedTokenizerFast
)
from transformers.trainer_utils import get_last_checkpoint
import evaluate

# === CONFIG ===
CSV_PATH = "./datasets/python_batch (3).csv"
BASE_MODEL_DIR = "scratch_llm_model"
BASE_TOKENIZER_DIR = "scratch_tokenizer"
MAX_INPUT = 256
MAX_OUTPUT = 128
EPOCHS = 3
BATCH_SIZE = 4
VOCAB_SIZE = 32000
LOG_FILE = "training_log.json"

# === Auto-versioning ===
def get_next_version_dir(base_dir):
    existing_versions = [
        int(re.search(rf"{base_dir}_v(\d+)", d).group(1))
        for d in os.listdir(".")
        if os.path.isdir(d) and re.match(rf"{base_dir}_v\d+", d)
    ]
    next_version = max(existing_versions, default=0) + 1
    return f"{base_dir}_v{next_version}"

MODEL_DIR = get_next_version_dir(BASE_MODEL_DIR)
TOKENIZER_DIR = MODEL_DIR.replace("model", "tokenizer")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
print(f"📁 Model will be saved to: {MODEL_DIR}")
print(f"📁 Tokenizer will be saved to: {TOKENIZER_DIR}")

# === Load dataset ===
df = pd.read_csv(CSV_PATH)[["input_text", "output_text"]].dropna()
dataset = Dataset.from_pandas(df)

# === Load tokenizer ===
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    BASE_TOKENIZER_DIR,
    model_max_length=MAX_INPUT,
    bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>"
)

# === Load or create model ===
if os.path.exists(os.path.join(BASE_MODEL_DIR, "pytorch_model.bin")):
    print("🔁 Loading base model from:", BASE_MODEL_DIR)
    model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL_DIR)
else:
    print("🚀 Creating model from scratch...")
    config = T5Config(
        vocab_size=VOCAB_SIZE,
        d_model=512,
        d_ff=2048,
        num_layers=6,
        num_heads=8,
        dropout_rate=0.1,
        eos_token_id=tokenizer.convert_tokens_to_ids("</s>"),
        pad_token_id=tokenizer.convert_tokens_to_ids("<pad>"),
        decoder_start_token_id=tokenizer.convert_tokens_to_ids("<pad>")
    )
    model = T5ForConditionalGeneration(config)

# === Tokenization ===
def tokenize(example):
    input_enc = tokenizer(
        example["input_text"], truncation=True, padding="max_length", max_length=MAX_INPUT
    )
    target_enc = tokenizer(
        example["output_text"], truncation=True, padding="max_length", max_length=MAX_OUTPUT
    )
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

dataset = dataset.map(tokenize, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# === Training args ===
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    eval_strategy="no",
    load_best_model_at_end=False,
    save_total_limit=3,
    logging_dir=os.path.join(MODEL_DIR, "logs"),
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    save_safetensors=False,
    resume_from_checkpoint=False
)

# === Training ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)

# === Evaluation (ROUGE) ===
rouge = evaluate.load("rouge")
print("Evaluating model...")
predictions, references = [], []

for row in df.sample(n=min(50, len(df)), random_state=42).itertuples():
    input_ids = tokenizer(
        row.input_text, return_tensors="pt", truncation=True,
        padding="max_length", max_length=MAX_INPUT, add_special_tokens=True
    ).input_ids
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=MAX_OUTPUT, num_beams=4)
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(decoded)
    references.append(row.output_text)

rouge_result = rouge.compute(predictions=predictions, references=references)
print("\n ROUGE Evaluation:")
for metric, score in rouge_result.items():
    print(f"{metric}: {round(score, 4)}")

# === Log Training Run ===
log_entry = {
    "timestamp": datetime.now().isoformat(),
    "model_version": MODEL_DIR,
    "tokenizer_version": TOKENIZER_DIR,
    "dataset": CSV_PATH,
    "total_samples": len(df),
    "epochs": EPOCHS,
    "rouge_scores": {k: round(v, 4) for k, v in rouge_result.items()}
}

# Append to log file
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, "r") as f:
        logs = json.load(f)
else:
    logs = []

logs.append(log_entry)
with open(LOG_FILE, "w") as f:
    json.dump(logs, f, indent=4)

print(f"\n Training complete. Logged to {LOG_FILE}")


📁 Model will be saved to: scratch_llm_model_v1
📁 Tokenizer will be saved to: scratch_llm_tokenizer_v1
🔁 Loading base model from: scratch_llm_model


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

🚦 Starting training...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,0.016200
20,0.020000
30,0.021600
40,0.019600
50,0.015600
60,0.011700
70,0.008200
80,0.008800
90,0.010200
100,0.011900


🧪 Evaluating model...

📊 ROUGE Evaluation:
rouge1: 0.8162
rouge2: 0.6487
rougeL: 0.8171
rougeLsum: 0.8168

✅ Training complete. Logged to training_log.json


In [4]:
# model loading and prediction. based on training. 


import torch
import pandas as pd
from transformers import T5ForConditionalGeneration, PreTrainedTokenizerFast

# === CONFIG ===
MODEL_DIR = "scratch_llm_model"
TOKENIZER_DIR = "scratch_tokenizer"
CSV_PATH = "./datasets/python_batch (3).csv"  # ✅ Use same or different dataset
MAX_INPUT = 256
MAX_OUTPUT = 128

# === Load tokenizer and model ===
print("📦 Loading model and tokenizer...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

# === Load dataset and pick one sample ===
df = pd.read_csv(CSV_PATH)[["input_text", "output_text"]].dropna()
# sample_row = df.sample(n=1, random_state=42).iloc[0]      this line is to select tatally random. row as the input_text while predicing. 
sample_row = df.iloc[4]   # for manually selecting any row.  index starts from 0, so 4 means 5th row.

input_text = sample_row.input_text.strip()
expected_output = sample_row.output_text.strip()

# === Tokenize input and generate output ===
inputs = tokenizer(
    input_text,
    return_tensors="pt",
    padding="max_length",
    max_length=MAX_INPUT,
    truncation=True,
    add_special_tokens=True  # ✅ Fixes missing characters like "G" in "Generates"
)

with torch.no_grad():
    generated_ids = model.generate(
        inputs["input_ids"],
        max_length=MAX_OUTPUT,
        num_beams=4,
        early_stopping=True
    )

predicted_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# === Print results ===
print("\n🧪 Input Text:\n", input_text)
print("\n✅ Expected Output:\n", expected_output)
print("\n🔮 Model Prediction:\n", predicted_output)


📦 Loading model and tokenizer...

🧪 Input Text:
 import sqlite3
conn = sqlite3.connect('test.db')
c = conn.cursor()
c.execute('CREATE TABLE IF NOT EXISTS users (id INTEGER, name TEXT)')
conn.commit()
conn.close()

✅ Expected Output:
 Creates a SQLite database and a table using the sqlite3 module.

🔮 Model Prediction:
 Creates a ite database and a table using the slite3 module.


Manually input the program for prediction.. 

In [9]:
import torch
from transformers import T5ForConditionalGeneration, PreTrainedTokenizerFast

# === CONFIG ===
MODEL_DIR = "scratch_llm_model"
TOKENIZER_DIR = "scratch_tokenizer"
MAX_INPUT = 256
MAX_OUTPUT = 128

# === Load model and tokenizer ===
print("📦 Loading model and tokenizer...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

print("🤖 Your model is ready! Type 'exit' to quit.")
print("👉 Example input: for Python summarization")
print('    def greet(name):\n        print("Hello " + name)\n')

while True:
    user_input = input("\n📝 Enter input_text:\n")
    if user_input.lower() == "exit":
        print("👋 Goodbye!")
        break

    # Tokenize and generate
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_INPUT)
    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            max_length=MAX_OUTPUT,
            num_beams=4,
            early_stopping=True
        )
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("\n🔮 Model Prediction:\n", prediction)




"""

📦 Loading model and tokenizer...
🤖 Your model is ready! Type 'exit' to quit.

👉 Example input: for Python summarization
    def greet(name):
        print("Hello " + name)


📝 Enter input_text:
def greet(name):
    print("Hello " + name)

🔮 Model Prediction:
Function to greet a person by name.



Type any code snippet (or input_text your model is trained on)

Type exit to quit.

"""

📦 Loading model and tokenizer...
🤖 Your model is ready! Type 'exit' to quit.
👉 Example input: for Python summarization
    def greet(name):
        print("Hello " + name)




📝 Enter input_text:
 def greet(name):         print("Hello " + name)



🔮 Model Prediction:
 enerates a  code for a  and saves it as an image.



📝 Enter input_text:
 import sqlite3 conn = sqlite3.connect('test.db') c = conn.cursor() c.execute('CREATE TABLE IF NOT EXISTS users (id INTEGER, name TEXT)') conn.commit() conn.close()



🔮 Model Prediction:
 Creates a ite database and a table using the slite3 module.



📝 Enter input_text:
 exit


👋 Goodbye!


'\n\n📦 Loading model and tokenizer...\n🤖 Your model is ready! Type \'exit\' to quit.\n\n👉 Example input: for Python summarization\n    def greet(name):\n        print("Hello " + name)\n\n\n📝 Enter input_text:\ndef greet(name):\n    print("Hello " + name)\n\n🔮 Model Prediction:\nFunction to greet a person by name.\n\n\n\nType any code snippet (or input_text your model is trained on)\n\nType exit to quit.\n\n'